Step 1: Input and Pair Embedding

In [ ]:
#!pip install torch einops

import torch
import torch.nn as nn
#from einops import rearrange

class InputEmbed(nn.Module):
    def __init__(self, seq_dim=4, d_model=128, max_len=512):
        super().__init__()
        self.embedding = nn.Embedding(seq_dim, d_model)
        self.positional_encoding = nn.Parameter(torch.randn(max_len, d_model))
    
    def forward(self, seq):
        x = self.embedding(seq) + self.positional_encoding[:seq.shape[1]]
        return x  # [batch, seq_len, d_model]


Step 2: Pairwise Embedding

In [ ]:
class PairEmbed(nn.Module):
    def __init__(self, d_model=128):
        super().__init__()
        self.linear = nn.Linear(d_model * 2, d_model)
    
    def forward(self, seq_feat):
        # Create pairwise feature: combine residue i and j
        seq_i = seq_feat.unsqueeze(2)  # [B, N, 1, D]
        seq_j = seq_feat.unsqueeze(1)  # [B, 1, N, D]
        pair = torch.cat([seq_i.repeat(1,1,seq_feat.size(1),1),
                          seq_j.repeat(1,seq_feat.size(1),1,1)], dim=-1)
        return self.linear(pair)  # [B, N, N, D]


Step 3: Evoformer Block (Simplified)

In [ ]:
class SimpleTransformerBlock(nn.Module):
    def __init__(self, d_model=128, heads=4):
        super().__init__()
        self.seq_attn = nn.TransformerEncoderLayer(d_model, heads)
        self.pair_proj = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.ReLU(),
            nn.Linear(d_model, d_model)
        )
    
    def forward(self, seq, pair):
        # Update sequence
        seq = self.seq_attn(seq)
        # Update pairwise (triangle updates not included here)
        pair = pair + self.pair_proj(pair)
        return seq, pair


Step 4: Structure Module

In [ ]:
class StructureModule(nn.Module):
    def __init__(self, d_model=128):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(d_model, 128),
            nn.ReLU(),
            nn.Linear(128, 3)  # Predict x, y, z
        )
    
    def forward(self, seq):
        return self.mlp(seq)  # [batch, seq_len, 3]


Step 5: Full Model

In [ ]:
class RNAFold(nn.Module):
    def __init__(self, n_blocks=4, d_model=128):
        super().__init__()
        self.embed = InputEmbed()
        self.pair_embed = PairEmbed(d_model)
        self.blocks = nn.ModuleList([SimpleTransformerBlock(d_model) for _ in range(n_blocks)])
        self.struct_module = StructureModule(d_model)
    
    def forward(self, seq):
        seq_feat = self.embed(seq)  # [B, N, D]
        pair_feat = self.pair_embed(seq_feat)  # [B, N, N, D]

        for block in self.blocks:
            seq_feat, pair_feat = block(seq_feat, pair_feat)

        coords = self.struct_module(seq_feat)  # [B, N, 3]
        return coords


Data Format for Training
Convert residues to indices: {'A': 0, 'U': 1, 'C': 2, 'G': 3}

In [ ]:
# Example batch
seq = torch.tensor([[0, 1, 2, 3, 0]])  # batch of 1, length 5
model = RNAFold()
coords = model(seq)  # output: [1, 5, 3]


Training Loop
Use a loss like RMSD or L2 loss between predicted and ground-truth coordinates.

In [ ]:
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

for epoch in range(100):
    for batch_seq, batch_coords in dataloader:
        optimizer.zero_grad()
        pred_coords = model(batch_seq)
        loss = criterion(pred_coords, batch_coords)
        loss.backward()
        optimizer.step()


Extras for Realism
If you want to mimic AlphaFold2 even closer:

Add triangle multiplicative updates in pairwise module.

Use invariant point attention to preserve geometric equivariance.

Add structure refinement loops.